In [15]:
!pip install pandas
!pip install Faker
!pip install numpy

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [16]:
import pandas as pd
import numpy as np

customer_batches = [3000, 5000, 7000]
total_customers = max(customer_batches)

transaction_df = pd.read_csv('../data/transactions.csv')
transaction_df.head()


,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


In [17]:
transaction_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  float64
 4   used_chip                       1000000 non-null  float64
 5   used_pin_number                 1000000 non-null  float64
 6   online_order                    1000000 non-null  float64
 7   fraud                           1000000 non-null  float64
dtypes: float64(8)
memory usage: 61.0 MB


In [18]:
import random
import uuid
from faker import Faker

fake = Faker()

In [19]:
country_fraud_rates = {
 'Nigeria': 0.2,
 'Russia': 0.2,
 'Brazil': 0.15,
 'India': 0.15,
 'Germany': 0.1,
 'France': 0.1,
 'US': 0.05,
 'UK': 0.05,
 'Singapore': 0.025,
 'Canada': 0.025,
 'Azerbaijan': 0.1,
 'Finland': 0.01
}

customers_df = pd.DataFrame()
customers_df['customer_id'] = np.arange(1, total_customers+1)
customers_df['name'] = [fake.name() for _ in range(total_customers)]
customers_df['age'] = np.random.randint(1, 60, size=total_customers)
customers_df['country'] = np.random.choice([*country_fraud_rates.keys()], total_customers)
customers_df['account_age'] = np.random.randint(0, 7, size=total_customers)
customers_df["income"] = np.random.randint(20000, 200000, size=len(customers_df))
risk_scores = np.concatenate([
    np.random.uniform(0.0,0.3,int(total_customers*0.7)),
    np.random.uniform(0.3,0.6,int(total_customers*0.2)),
    np.random.uniform(0.6,1.0,int(total_customers*0.1))
])

np.random.shuffle(risk_scores)
customers_df['risk_score'] = risk_scores

start_idx = 0
for i, end_idx in enumerate(customer_batches):
    # first batch will be used for inital ingestion
    customer_batch = customers_df[start_idx:end_idx]
    if start_idx == 0:
        customer_batch.to_csv("../data/customers_initial.csv")
    else:
        customer_batch.to_csv(f"../data/customers_batch{i}.csv")
    start_idx = end_idx

customers_df.head()

,customer_id,name,age,country,account_age,income,risk_score
0,1,John Miller,9,India,5,65992,0.254910
1,2,Krista Zimmerman MD,6,Singapore,3,156849,0.221930
2,3,Pam Richardson,30,India,0,66445,0.216870
3,4,Ronald Pruitt,56,Azerbaijan,3,153656,0.022504
4,5,Mary Gonzales,48,Finland,4,130777,0.303144


In [ ]:
"""
    Separate the transactions data into sections with the chunks of customer id so the future customer ids don't appear in the old transaction data.
    We are also adding customer_id so that we can connect customer data with the transaction,
    which will helps us to make better predictions in ML part.
"""
# We use activity_weights to create customers that are highly active to make the dataset more realistic
customer_ids = np.arange(1, total_customers+1)
activiy_weights = np.random.zipf(2, total_customers)
activiy_weights = activiy_weights / activiy_weights.sum()
total_fruad_rate = sum([*country_fraud_rates.values()])
fraud_weights = [v / total_fruad_rate for v in country_fraud_rates.values()]
country_customer_map = customers_df.groupby('country')['customer_id'].apply(list).to_dict()

fraud_mask = transaction_df['fraud'] == 1
nonfraud_mask = transaction_df['fraud'] == 0

# Random customer_id assigment for nonfraud transactions
# + based on activity weights(some users are active while some aren't)
transaction_df.loc[nonfraud_mask, 'customer_id'] = np.random.choice(
    customer_ids,
    size=nonfraud_mask.sum(),
    p=activiy_weights
)

# Create country names based on fraud rates for each country
fraud_countries = np.random.choice(
    list(country_fraud_rates.keys()),
    size=fraud_mask.sum(),
    p=fraud_weights
)

# Choose random id from <country: customer_id list>
fraud_customer_ids = [
    np.random.choice(country_customer_map[c])
        for c in fraud_countries
]

transaction_df.loc[fraud_mask, 'customer_id'] = fraud_customer_ids
transaction_df['customer_id'] = transaction_df['customer_id'].astype('int')
transaction_df['id'] = [str(uuid.uuid4()) for _ in range(len(transaction_df))]

from datetime import datetime, timedelta

# Base start date
start_date = datetime(2024, 1, 1)

n = len(transaction_df)

# Generate random days
days_offset = np.random.randint(0, 90, size=n)  # 3 months of data

# Generate realistic hours
# More weight to daytime hours
hours = np.array(range(24))
hour_weights = np.array([
    1,1,1,1,1,   # 0–4 (very low)
    2,3,5,7,9,   # 5–9 (morning rise)
    10,10,10,9,8,# 10–14 (peak)
    7,8,9,10,9,  # 15–19 (evening peak)
    7,5,3,2      # 20–23 (decline)
])
hour_weights = hour_weights / hour_weights.sum()

random_hours = np.random.choice(hours, size=n, p=hour_weights)

# Minutes + seconds
minutes = np.random.randint(0, 60, size=n)
seconds = np.random.randint(0, 60, size=n)

# Combine into timestamps
timestamps = [
    start_date + timedelta(days=int(d), hours=int(h), minutes=int(m), seconds=int(s))
    for d, h, m, s in zip(days_offset, random_hours, minutes, seconds)
]

transaction_df["timestamp"] = timestamps

start_idx = 1
for i, end_idx in enumerate(customer_batches):
    transaction_batch = transaction_df[(transaction_df['customer_id'] >= start_idx) & (transaction_df['customer_id'] < end_idx)]
    if start_idx == 1:
        transaction_batch.to_csv("../data/transactions_initial.csv")
    else:
        transaction_batch.to_csv(f"../data/transactions_stream{i}.csv")
    start_idx = end_idx

transaction_df.head()


In [21]:
fraud_messages = [
    "Someone used my card without permission",
    "I didn't make this transaction",
    "There is a faudulent charge",
    "My account was hacked"
]

normal_messages = [
    "How do I reset my password?",
    "Thank you for your help",
    "I want to change my address",
    "How long does a transfer take?"
]

logs = []
fraud_customers = transaction_df.loc[
    transaction_df["fraud"] == 1, "customer_id"
].unique()

for i in range(total_customers):
    if random.random() < 0.15:
        # fraud
        msg = np.random.choice(fraud_messages)
        # 70% of messages about fraud is actually fraud
        if random.random() < 0.7:
            chat_customer_id = np.random.choice(fraud_customers)
        #other 30% is random
        else:
            chat_customer_id = np.random.choice(customer_ids)
    else:
        # normal
        msg = np.random.choice(normal_messages)
        chat_customer_id = np.random.choice(
            customer_ids
        )

    logs.append({
        "id": str(uuid.uuid4()),
        "customer_id": chat_customer_id,
        "timestamp": fake.date_time_this_year(),
        "message": msg
    })

chat_logs_df = pd.DataFrame(logs)
start_idx = 1
for i, end_idx in enumerate(customer_batches):
    chat_logs_batch = chat_logs_df[(chat_logs_df['customer_id'] >= start_idx) & (chat_logs_df['customer_id'] < end_idx)]
    if start_idx == 1:
        chat_logs_batch.to_csv("../data/chat_logs_initial.csv")
    else:
        chat_logs_batch.to_csv(f"../data/chat_logs{i}.csv")
    start_idx = end_idx
chat_logs_df.head()

,id,customer_id,timestamp,message
0,6d8831ed-acd7-4df3-b185-85a937254de6,6812,2026-01-05 07:01:00.458349,I want to change my address
1,d64d30d5-446b-4cdb-89b4-594358eddefa,1876,2026-01-28 12:10:39.782766,I want to change my address
2,efac917c-e341-4ddc-a690-93e4ab03ba61,5602,2026-03-18 06:03:25.243000,I didn't make this transaction
3,4c9236f9-3389-4c8f-a7ea-72bd875c4808,1238,2026-02-15 04:27:24.477939,How long does a transfer take?
4,8161daba-67aa-48e5-8ff4-9b28c96770c4,2855,2026-01-02 16:49:44.954944,How long does a transfer take?
